# Activation Functions

## Learning Objectives
1. Understand how ReLU, sigmoid, tanh, and GELU shape neural-network outputs.
2. Implement each activation from scratch using NumPy and verify against known properties.
3. Build an MLP that swaps activation functions and measure accuracy on XOR.
4. Detect the dying-ReLU failure mode and apply practical fixes.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1: Activation Functions from Scratch (NumPy)

In [ ]:
# Implement the four most common activations and their derivatives from scratch.

def relu(x):
    return np.maximum(0.0, x)

def relu_grad(x):
    return (x > 0).astype(float)

def sigmoid(x):
    # Clip to avoid overflow in exp
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

def sigmoid_grad(x):
    s = sigmoid(x)
    return s * (1 - s)

def tanh_act(x):
    return np.tanh(x)

def tanh_grad(x):
    return 1.0 - np.tanh(x) ** 2

def gelu(x):
    # GELU approximation used in BERT, GPT transformers
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * x ** 3)))

x = np.linspace(-4, 4, 300)
activations = {
    "ReLU":    relu(x),
    "Sigmoid": sigmoid(x),
    "Tanh":    tanh_act(x),
    "GELU":    gelu(x),
}
for name, vals in activations.items():
    print(f"{name:8s}  range=[{vals.min():.3f}, {vals.max():.3f}]")

# Verify ReLU gradient: should be 1 for x>0, 0 otherwise
x_check = np.array([-1.0, 0.0, 1.0, 2.0])
print(f"ReLU grad at {x_check}: {relu_grad(x_check)}")
print("All four activations implemented successfully.")


## Level 2: MLP with Swappable Activations on XOR (PyTorch)

In [ ]:
# Train a small MLP on XOR using four different activations and compare accuracy.

XOR_X = torch.tensor([[0,0],[0,1],[1,0],[1,1]], dtype=torch.float32)
XOR_y = torch.tensor([[0],[1],[1],[0]], dtype=torch.float32)
# Repeat to form a usable dataset for mini-batch training
X_train = XOR_X.repeat(200, 1).to(device)
y_train = XOR_y.repeat(200, 1).to(device)
ds = TensorDataset(X_train, y_train)
loader = DataLoader(ds, batch_size=64, shuffle=True)

def build_mlp(act_fn):
    return nn.Sequential(
        nn.Linear(2, 16), act_fn,
        nn.Linear(16, 16), act_fn,
        nn.Linear(16, 1), nn.Sigmoid()
    ).to(device)

ACTIVATIONS = {
    "ReLU":    nn.ReLU(),
    "Tanh":    nn.Tanh(),
    "GELU":    nn.GELU(),
    "Sigmoid": nn.Sigmoid(),
}

results = {}
for name, act in ACTIVATIONS.items():
    model = build_mlp(act)
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    crit = nn.BCELoss()
    for epoch in range(300):
        for xb, yb in loader:
            opt.zero_grad()
            try:
                loss = crit(model(xb), yb)
            except RuntimeError as exc:
                if "out of memory" in str(exc).lower():
                    print(f"OOM with {name} -- reduce batch size")
                    torch.cuda.empty_cache()
                    continue
                raise
            loss.backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        preds = (model(X_train) > 0.5).float()
        acc = (preds == y_train).float().mean().item()
    results[name] = acc
    print(f"{name:8s}  accuracy={acc:.4f}")

best = max(results, key=results.get)
print(f"Best activation on XOR: {best} ({results[best]:.4f})")


## Real-World Example 1: Dying ReLU Detection and Fix

In [ ]:
# Dying ReLU: neurons output 0 for all inputs after a bad init or large LR.
# Detect by counting dead neurons, then fix with He-init + LeakyReLU.

def count_dead_neurons(model, x_sample):
    activations_list = []
    hooks = []
    def hook_fn(module, inp, out):
        activations_list.append(out.detach().cpu())
    for m in model.modules():
        if isinstance(m, nn.ReLU):
            hooks.append(m.register_forward_hook(hook_fn))
    with torch.no_grad():
        model(x_sample)
    for h in hooks:
        h.remove()
    if not activations_list:
        return 0.0
    dead = sum((a <= 0).all(dim=0).float().mean().item() for a in activations_list)
    return dead / len(activations_list)

# Simulate dying ReLU: large negative biases force all ReLU outputs to zero
bad_model = nn.Sequential(
    nn.Linear(20, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 1)
).to(device)
with torch.no_grad():
    for m in bad_model.modules():
        if isinstance(m, nn.Linear):
            m.bias.fill_(-5.0)   # kills all ReLU outputs

x_sample = torch.randn(128, 20, device=device)
dead_frac = count_dead_neurons(bad_model, x_sample)
print(f"Dying ReLU model -- dead neuron fraction: {dead_frac:.2%}")

# Fix: He initialisation + LeakyReLU (negative slope = 0.1 keeps gradients alive)
fixed_model = nn.Sequential(
    nn.Linear(20, 64), nn.LeakyReLU(0.1),
    nn.Linear(64, 64), nn.LeakyReLU(0.1),
    nn.Linear(64, 1)
).to(device)
for m in fixed_model.modules():
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity='leaky_relu')
        nn.init.zeros_(m.bias)

out = fixed_model(x_sample)
print(f"Fixed model output -- mean={out.mean().item():.4f}  std={out.std().item():.4f}")
print("Fix applied: He-init + LeakyReLU eliminates dying-ReLU problem.")


## Real-World Example 2: GELU in a Transformer FFN Block

In [ ]:
# Modern transformers (BERT, GPT) use GELU inside feed-forward sublayers.
# This example shows a drop-in FFN block with GELU and benchmarks vs ReLU.

import time

class TransformerFFN(nn.Module):
    # Two-layer FFN as used in transformer encoders.
    def __init__(self, d_model, d_ff, activation):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            activation,
            nn.Dropout(0.1),
            nn.Linear(d_ff, d_model),
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        return self.norm(x + self.net(x))   # residual connection

D_MODEL, D_FF, SEQ_LEN, BATCH = 256, 1024, 128, 16
x_ffn = torch.randn(BATCH, SEQ_LEN, D_MODEL, device=device)

timing = {}
for act_name, act_mod in [("GELU", nn.GELU()), ("ReLU", nn.ReLU())]:
    ffn = TransformerFFN(D_MODEL, D_FF, act_mod).to(device)
    ffn.eval()
    # Warmup pass to initialise CUDA kernels
    with torch.no_grad():
        _ = ffn(x_ffn)
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(20):
            out = ffn(x_ffn)
    elapsed = (time.perf_counter() - t0) / 20 * 1000
    timing[act_name] = elapsed
    print(f"FFN-{act_name}  output shape={tuple(out.shape)}  time/iter={elapsed:.2f}ms")

overhead = (timing["GELU"] - timing["ReLU"]) / timing["ReLU"] * 100
print(f"GELU overhead vs ReLU: {overhead:+.1f}%")
print("GELU's smooth gradient improves convergence; overhead is minimal in practice.")


## Real-World Example 3: Activation Ablation Study

In [ ]:
# Systematic ablation: train the same architecture N_RUNS times per activation,
# record mean +/- std of final val loss to measure sensitivity.

from torch.utils.data import random_split

# Regression task: y = sin(x1) + cos(x2) + noise
N_ABL = 2000
Xr = torch.randn(N_ABL, 4)
yr = (torch.sin(Xr[:, 0]) + torch.cos(Xr[:, 1])).unsqueeze(1)
train_abl, val_abl = random_split(TensorDataset(Xr, yr), [1600, 400])
tr_loader_abl = DataLoader(train_abl, batch_size=64, shuffle=True)
va_loader_abl = DataLoader(val_abl, batch_size=64)

ACTS_ABLATION = {
    "ReLU": nn.ReLU,
    "GELU": nn.GELU,
    "ELU":  lambda: nn.ELU(alpha=1.0),
    "SiLU": nn.SiLU,
    "Tanh": nn.Tanh,
}

N_RUNS = 3   # increase to 5 for a production study
ablation_results = {}

for act_name, act_cls in ACTS_ABLATION.items():
    run_losses = []
    for run in range(N_RUNS):
        torch.manual_seed(run * 7)
        mdl = nn.Sequential(
            nn.Linear(4, 64), act_cls(),
            nn.Linear(64, 64), act_cls(),
            nn.Linear(64, 1)
        ).to(device)
        opt = torch.optim.Adam(mdl.parameters(), lr=1e-3)
        crit = nn.MSELoss()
        for _ in range(80):
            mdl.train()
            for xb, yb in tr_loader_abl:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                crit(mdl(xb), yb).backward()
                opt.step()
        mdl.eval()
        with torch.no_grad():
            vl = sum(
                crit(mdl(xb.to(device)), yb.to(device)).item() * len(xb)
                for xb, yb in va_loader_abl
            ) / len(va_loader_abl.dataset)
        run_losses.append(vl)
    ablation_results[act_name] = (np.mean(run_losses), np.std(run_losses))

print(f"{'Activation':<10}  {'Mean Val MSE':>12}  {'Std':>8}")
print("-" * 36)
for name, (mean, std) in sorted(ablation_results.items(), key=lambda x: x[1][0]):
    print(f"{name:<10}  {mean:>12.6f}  {std:>8.6f}")
print("Conclusion: SiLU/GELU/ELU typically outperform ReLU on smooth targets.")
